# Cartes par quart de département

Génère, pour chaque département choisi, 4 cartes (un quart NO/NE/SO/SE de son emprise) superposant les isochrones Géofer et la densité de population des carreaux INSEE 200x200 m, avec les zones non desservies mises en évidence — même logique que `app.py`, réutilisée telle quelle.

Chaque quart est exporté en PNG (impression/aperçu rapide) et en HTML interactif, dans `Output/`.

In [ ]:
import os
import sys

import contextily as cx
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.validation import make_valid

sys.path.insert(0, ".")
import app

OUTPUT_DIR = "Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CARREAUX_CMAP = LinearSegmentedColormap.from_list("carreaux", app.CARREAUX_COLOR_SCALE)

# Codes INSEE des départements à traiter. Pour un export national complet :
# DEPARTEMENTS = app.load_departements()["code"].tolist()
DEPARTEMENTS = ["51", "08", "02"]  # Marne, Ardennes, Aisne

In [ ]:
def decouper_en_quadrants(dept_geom):
    """Découpe l'emprise d'un département en 4 quarts (NO/NE/SO/SE), clippés à sa forme réelle."""
    minx, miny, maxx, maxy = dept_geom.bounds
    midx, midy = (minx + maxx) / 2, (miny + maxy) / 2
    boites = {
        "NO": box(minx, midy, midx, maxy),
        "NE": box(midx, midy, maxx, maxy),
        "SO": box(minx, miny, midx, midy),
        "SE": box(midx, miny, maxx, midy),
    }
    quadrants = {}
    for nom, boite in boites.items():
        geom = dept_geom.intersection(boite)
        if not geom.is_empty:
            quadrants[nom] = geom
    return quadrants

In [ ]:
def donnees_quadrant(quadrant_geom, quadrant_key, gares, insee_path):
    """Gares, isochrones et carreaux INSEE (avec statut desservi/non desservi) pour un quart de département.

    quadrant_key doit être unique par quart (ex. "51_NO") : app.load_insee_carreaux est mise en cache
    par Streamlit sur cet identifiant, pas sur la géométrie elle-même.
    """
    bounds = quadrant_geom.bounds
    gares_quad = gares[
        gares["wgs84Lon"].between(bounds[0], bounds[2]) & gares["wgs84Lat"].between(bounds[1], bounds[3])
    ]
    codes_quad = set(gares_quad["codeUic"])

    isochrones_quad = {}
    for mode, (path, _) in app.ISOCHRONE_FILES.items():
        full = app.load_isochrones(path)
        isochrones_quad[mode] = full[full["code_uic"].isin(codes_quad)]

    carreaux = app.load_insee_carreaux(insee_path, quadrant_geom, quadrant_key)
    if not carreaux.empty:
        served_geoms = [geom for gdf in isochrones_quad.values() for geom in gdf.geometry]
        union_geom = unary_union(served_geoms) if served_geoms else None
        carreaux["desservi"] = carreaux.intersects(union_geom) if union_geom is not None else False

    return gares_quad, isochrones_quad, carreaux

In [ ]:
def rendre_png(quadrant_geom, gares_quad, isochrones_quad, carreaux, titre, chemin_sortie, color_field="pop"):
    fig, ax = plt.subplots(figsize=(10, 10))

    if not carreaux.empty:
        carreaux_3857 = carreaux.to_crs(3857)
        desservi = carreaux_3857[carreaux_3857["desservi"]]
        non_desservi = carreaux_3857[~carreaux_3857["desservi"]]
        if not desservi.empty:
            desservi.plot(ax=ax, color="#c8ced6", alpha=0.35, linewidth=0)
        if not non_desservi.empty:
            non_desservi.plot(
                ax=ax, column=color_field, cmap=CARREAUX_CMAP, alpha=0.92, linewidth=0,
                legend=True, legend_kwds={"label": color_field, "shrink": 0.6},
            )

    for mode, gdf in isochrones_quad.items():
        if gdf.empty:
            continue
        _, couleur = app.ISOCHRONE_FILES[mode]
        gdf.to_crs(3857).plot(ax=ax, facecolor=couleur, edgecolor=couleur, alpha=0.3, linewidth=1.5)

    if not gares_quad.empty:
        gares_pts = gpd.GeoDataFrame(
            gares_quad,
            geometry=gpd.points_from_xy(gares_quad["wgs84Lon"], gares_quad["wgs84Lat"]),
            crs="EPSG:4326",
        ).to_crs(3857)
        gares_pts.plot(ax=ax, color="#581012", marker="^", markersize=40, zorder=5)

    quadrant_3857 = gpd.GeoSeries([quadrant_geom], crs="EPSG:4326").to_crs(3857).iloc[0]
    minx, miny, maxx, maxy = quadrant_3857.bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
    ax.set_axis_off()
    ax.set_title(titre, fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(chemin_sortie, dpi=150)
    plt.close(fig)

In [ ]:
def rendre_html(quadrant_geom, gares_quad, isochrones_quad, carreaux, color_field, color_label, chemin_sortie):
    bounds = quadrant_geom.bounds
    centre = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
    m = folium.Map(location=centre, tiles="CartoDB positron", prefer_canvas=True)

    if not carreaux.empty:
        unserved = carreaux[~carreaux["desservi"]]
        source_echelle = unserved if not unserved.empty else carreaux
        colormap = folium.LinearColormap(
            colors=app.CARREAUX_COLOR_SCALE,
            vmin=float(source_echelle[color_field].min()),
            vmax=float(source_echelle[color_field].max()),
            caption=f"{color_label} — carreaux non desservis",
        )

        def style_carreau(feature, cf=color_field, cm=colormap):
            if feature["properties"]["desservi"]:
                return {"fillColor": "#c8ced6", "color": "#9aa3af", "weight": 0, "fillOpacity": 0.35}
            return {"fillColor": cm(feature["properties"][cf]), "color": "#581012", "weight": 0, "fillOpacity": 0.92}

        folium.GeoJson(
            carreaux,
            style_function=style_carreau,
            tooltip=folium.GeoJsonTooltip(
                fields=["pop", "niveau_vie", "taux_pauvrete", "part_65p", "desservi"],
                aliases=["Population", "Revenu moyen (€)", "Taux de pauvreté (%)", "Part 65 ans+ (%)", "Desservi"],
                localize=True,
            ),
        ).add_to(m)
        colormap.add_to(m)

    for mode, gdf in isochrones_quad.items():
        if gdf.empty:
            continue
        _, couleur = app.ISOCHRONE_FILES[mode]
        folium.GeoJson(
            gdf,
            name=mode,
            style_function=lambda f, c=couleur: {"color": c, "weight": 2, "fill": True, "fillColor": c, "fillOpacity": 0.3},
        ).add_to(m)

    for _, gare in gares_quad.iterrows():
        folium.Marker(
            [gare["wgs84Lat"], gare["wgs84Lon"]],
            tooltip=gare["nomGare"],
            icon=folium.Icon(color="darkred", icon="train", prefix="fa"),
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    m.save(chemin_sortie)

In [ ]:
gares = app.load_gares()
departements = app.load_departements()
insee_path = app.get_insee_local_path()

for dept_code in DEPARTEMENTS:
    dept = departements[departements["code"] == dept_code].iloc[0]
    quadrants = decouper_en_quadrants(dept.geometry)

    for nom_quad, quad_geom in quadrants.items():
        quadrant_key = f"{dept_code}_{nom_quad}"
        gares_quad, isochrones_quad, carreaux = donnees_quadrant(quad_geom, quadrant_key, gares, insee_path)

        titre = f"{dept['nom']} ({dept_code}) — {nom_quad}"
        base_nom = f"{dept_code}_{nom_quad}"

        rendre_png(quad_geom, gares_quad, isochrones_quad, carreaux, titre, f"{OUTPUT_DIR}/{base_nom}.png")
        rendre_html(quad_geom, gares_quad, isochrones_quad, carreaux, "pop", "Population", f"{OUTPUT_DIR}/{base_nom}.html")

        print(f"{base_nom} : {len(carreaux)} carreaux, {len(gares_quad)} gares")

print("Terminé —", OUTPUT_DIR)